# Dataset Download and Quick Inspection

1. Download raw benchmark datasets (Yoochoose and Diginetica) from Kaggle.
2. Store untouched artifacts in `data/raw/`.
3. Preview table information (fields, shape, sample rows).
4. Write download manifest with source, date, file names, and sizes.

Requirements:
- Install python packages from `requirements.txt`.
- Kaggle access configured for your environment (required by `kagglehub`).

In [10]:
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import kagglehub
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DATE_UTC = datetime.now(timezone.utc).isoformat()

In [11]:
def find_file(root_dir, filename):
    exact_matches = list(root_dir.rglob(filename))
    if exact_matches:
        return exact_matches[0]

    filename_lower = filename.lower()
    for path in root_dir.rglob("*"):
        if path.is_file() and path.name.lower() == filename_lower:
            return path

    return None


DATASETS = [
    {
        "name": "diginetica",
        "kaggle_dataset": "profalbusdumbledore/diginetica-dataset",
        "tables": [
            {
                "table_name": "item_views",
                "filename": "train-item-views.csv",
                "has_header": True,
                "fields": [
                    "session_id",
                    "user_id",
                    "item_id",
                    "timeframe",
                    "eventdate",
                ],
            },
            {
                "table_name": "product_categories",
                "filename": "product-categories.csv",
                "has_header": True,
                "fields": ["item_id", "category_id"],
            },
        ],
    },
    {
        "name": "yoochoose",
        "kaggle_dataset": "phhasian0710/yoochoose",
        "tables": [
            {
                "table_name": "clicks",
                "filename": "yoochoose-clicks.dat",
                "has_header": False,
                "fields": ["session_id", "timestamp", "item_id", "category"],
            },
            {
                "table_name": "buys",
                "filename": "yoochoose-buys.dat",
                "has_header": False,
                "fields": ["session_id", "timestamp", "item_id", "price", "quantity"],
            },
        ],
    },
]

manifest = []
PREPARED_TABLES = []

for dataset in DATASETS:
    print(f"Dataset: {dataset['name']}")

    kaggle_cache_path = Path(kagglehub.dataset_download(dataset["kaggle_dataset"]))

    raw_dataset_dir = RAW_DIR / dataset["name"]
    raw_dataset_dir.mkdir(parents=True, exist_ok=True)

    dataset_file_names = []
    dataset_file_sizes = {}

    for table in dataset["tables"]:
        source_path = find_file(kaggle_cache_path, table["filename"])
        copied_path = None

        if source_path is None:
            print(f"Missing file in Kaggle dataset: {table['filename']}")
        else:
            copied_path = raw_dataset_dir / table["filename"]
            shutil.copy2(source_path, copied_path)
            rel_path = str(copied_path.relative_to(RAW_DIR))
            dataset_file_names.append(rel_path)
            dataset_file_sizes[rel_path] = copied_path.stat().st_size

        PREPARED_TABLES.append(
            {
                "dataset": dataset["name"],
                "table_name": table["table_name"],
                "fields": table["fields"],
                "has_header": table["has_header"],
                "path": copied_path,
            }
        )

    manifest.append(
        {
            "dataset": dataset["name"],
            "kaggle_dataset": dataset["kaggle_dataset"],
            "download_date_utc": DOWNLOAD_DATE_UTC,
            "raw_file_names": dataset_file_names,
            "raw_file_sizes_bytes": dataset_file_sizes,
        }
    )

manifest_path = RAW_DIR / "download_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

Dataset: diginetica
Dataset: yoochoose


806

In [12]:
def get_row_count(file_path, has_header):
    result = subprocess.run(
        ["wc", "-l", str(file_path)], capture_output=True, text=True, check=True
    )
    line_count = int(result.stdout.strip().split()[0])
    if has_header and line_count > 0:
        return line_count - 1
    return line_count


def preview_table(table_info):
    if table_info["path"] is None:
        print("\n" + "-" * 80)
        print(
            f"{table_info['dataset']} / {table_info['table_name']}: file not found after extraction."
        )
        return

    file_path = Path(table_info["path"])
    fields = table_info["fields"]
    has_header = table_info["has_header"]

    if has_header:
        df_head = pd.read_csv(file_path, nrows=5)
        actual_fields = list(df_head.columns)
    else:
        df_head = pd.read_csv(file_path, header=None, names=fields, nrows=5)
        actual_fields = fields

    rows = get_row_count(file_path, has_header)
    cols = len(actual_fields)

    print("\n" + "-" * 80)
    print(f"Dataset: {table_info['dataset']}")
    print(f"Table: {table_info['table_name']}")
    print(f"Fields ({cols}): {actual_fields}")
    print(f"Shape: ({rows}, {cols})")
    print("Sample rows:")
    print(df_head)


for table in PREPARED_TABLES:
    preview_table(table)


--------------------------------------------------------------------------------
Dataset: diginetica
Table: item_views
Fields (1): ['sessionId;userId;itemId;timeframe;eventdate']
Shape: (1235380, 1)
Sample rows:
  sessionId;userId;itemId;timeframe;eventdate
0                1;NA;81766;526309;2016-05-09
1               1;NA;31331;1031018;2016-05-09
2                1;NA;32118;243569;2016-05-09
3                  1;NA;9654;75848;2016-05-09
4               1;NA;32627;1112408;2016-05-09

--------------------------------------------------------------------------------
Dataset: diginetica
Table: product_categories
Fields (1): ['itemId;categoryId']
Shape: (184047, 1)
Sample rows:
  itemId;categoryId
0       139578;1096
1       417975;1096
2       291805;1096
3       396921;1096
4       159257;1096

--------------------------------------------------------------------------------
Dataset: yoochoose
Table: clicks
Fields (4): ['session_id', 'timestamp', 'item_id', 'category']
Shape: (33003944, 4